In [1]:
import numpy as np, pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_parquet("../data/processed/reviews_clean.parquet")
embeddings = np.load("../data/processed/embeddings.npy")
docs = df["text"].tolist()

umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, metric="euclidean",
                        cluster_selection_method="eom", prediction_data=True)
vectorizer = CountVectorizer(stop_words="english", min_df=10,
                             ngram_range=(1, 2))

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    verbose=True,
)
topics, probs = topic_model.fit_transform(docs, embeddings)

print(topic_model.get_topic_info().head(20))
print("Outliers:", (np.array(topics) == -1).mean())

2026-10-20 12:48:44,440 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-20 12:50:02,628 - BERTopic - Dimensionality - Completed ✓
2026-10-20 12:50:02,630 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-20 12:50:08,538 - BERTopic - Cluster - Completed ✓
2026-10-20 12:50:08,549 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-20 12:50:15,518 - BERTopic - Representation - Completed ✓


    Topic  Count                                   Name  \
0      -1  22623               -1_book_like_album_great   
1       0   4061                0_book_read_story_books   
2       1   3002             1_movie_film_movies_acting   
3       2   1178               2_dvd_bluray_disc_player   
4       3    880                 3_album_cd_voice_songs   
5       4    809                    4_toy_loves_son_old   
6       5    799             5_game_games_graphics_play   
7       6    607          6_item_product_order_received   
8       7    587       7_book_programming_examples_text   
9       8    546            8_bible_god_christian_jesus   
10      9    540                    9_pan_oven_pans_lid   
11     10    472            10_taste_tea_flavor_product   
12     11    463                 11_jazz_blues_album_cd   
13     12    446       12_condition_pages_book_received   
14     13    435             13_skin_hair_product_smell   
15     14    432      14_season_episodes_series_seasons 

In [2]:
topic_model.save("../models/bertopic_v1", serialization="safetensors",
                 save_ctfidf=True,
                 save_embedding_model="all-MiniLM-L6-v2")
df["topic_v1"] = topics
df[["review_id", "topic_v1"]].to_parquet(
    "../data/processed/topics_v1.parquet", index=False)

In [3]:
new_topics = topic_model.reduce_outliers(
    docs, topics, strategy="embeddings", embeddings=embeddings
)
topic_model.update_topics(docs, topics=new_topics,
                          vectorizer_model=vectorizer)
print("Outliers now:", (np.array(new_topics) == -1).mean())

2026-10-20 12:50:44,757 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Outliers now: 0.0


In [4]:
df["topic_v1_reduced"] = new_topics
df[["review_id", "topic_v1_reduced"]].to_parquet(
    "../data/processed/topics_v1_reduced.parquet", index=False)

In [5]:
hdbscan_model = HDBSCAN(min_cluster_size=30, min_samples=5,
                        metric="euclidean", cluster_selection_method="eom",
                        prediction_data=True)

In [6]:
import pandas as pd
t = pd.read_parquet("../data/processed/topics_v1.parquet")
print((t["topic_v1"] == -1).mean())          # expect ~0.45
print(t["topic_v1"].nunique() - 1)           # expect 108

0.45246
109


In [7]:
t = pd.read_parquet("../data/processed/topics_v1.parquet")
real = sorted(set(t["topic_v1"]) - {-1})
print(len(real), min(real), max(real))   # expect 109, 0, 108
print(t["topic_v1"].isna().sum())        # expect 0


109 0 108
0


In [8]:
print(len(topic_model.get_topic_info()) - 1)

108


In [ ]:
topic_model = BERTopic.load("../models/bertopic_v1")
new_topics = topic_model.reduce_outliers(
    docs, topics, strategy="embeddings", embeddings=embeddings
)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=vectorizer)
print("Outliers now:", (np.array(new_topics) == -1).mean())

df["topic_v1_reduced"] = new_topics
df[["review_id", "topic_v1_reduced"]].to_parquet(
    "../data/processed/topics_v1_reduced.parquet", index=False)